# 07 - Retrieval-Augmented Generation (RAG) -- Concept Only (OPTIONAL)

---

> **This notebook is OPTIONAL and CONCEPT-FOCUSED.** It explains the RAG pattern and provides a lightweight simulation using in-memory documents and cosine similarity. **No API calls or large models required.**

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain the **RAG pipeline**: Query -> Retrieve -> Augment prompt -> Generate
- Understand **why RAG exists**: LLM hallucinations and knowledge cutoff
- Describe the key components: **document store, embeddings, retriever, generator**
- Explain **chunking strategies** for splitting documents
- Understand the concept of **vector databases**
- Implement a **RAG simulation** with in-memory cosine similarity retrieval

## Prerequisites

- Completed Notebook 05 (embeddings and semantic search)
- Understanding of cosine similarity and sentence embeddings
- No external libraries beyond PyTorch, NumPy, and scikit-learn

## Table of Contents

1. [The Problem: Why RAG?](#1-the-problem-why-rag)
2. [RAG Architecture Overview](#2-rag-architecture-overview)
3. [Component 1: Document Store and Chunking](#3-component-1-document-store-and-chunking)
4. [Component 2: Embeddings and Vector Store](#4-component-2-embeddings-and-vector-store)
5. [Component 3: Retriever](#5-component-3-retriever)
6. [Component 4: Generator (Concept)](#6-component-4-generator-concept)
7. [Code: RAG Simulation](#7-code-rag-simulation)
8. [Vector Databases (Concept)](#8-vector-databases-concept)
9. [Exercise: Design a RAG Pipeline on Paper](#9-exercise-design-a-rag-pipeline-on-paper)
10. [Common Mistakes & Debugging Tips](#10-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

print(f"PyTorch version: {torch.__version__}")

---
## 1. The Problem: Why RAG?

Large Language Models (LLMs) have two fundamental limitations:

### Hallucination

LLMs can generate **plausible-sounding but factually incorrect** information:

- *"Albert Einstein won the Nobel Prize in Physics in 1905 for his theory of relativity."*
  - Incorrect: He won in 1921, for the photoelectric effect.
- The model is not "lying" -- it is generating the most probable continuation based on training data patterns.

### Knowledge Cutoff

LLMs only know what was in their training data:

- Cannot answer questions about events after their training cutoff date
- Cannot access private/proprietary documents
- Cannot reference the latest documentation or research

### RAG Solution

**Retrieval-Augmented Generation** addresses both problems by:

1. **Retrieving** relevant documents from a knowledge base
2. **Augmenting** the prompt with retrieved context
3. **Generating** a response grounded in the retrieved information

The LLM now has access to **external, up-to-date, factual information** at generation time.

---
## 2. RAG Architecture Overview

```
User Query
    |
    v
+-------------------+
|  1. EMBED QUERY   |   Convert query to a vector
+-------------------+
    |
    v
+-------------------+     +-------------------+
|  2. RETRIEVE      |<--->| Vector Store      |
|  Find top-k docs  |     | (pre-embedded     |
+-------------------+     |  document chunks)  |
    |                     +-------------------+
    v
+-------------------+
|  3. AUGMENT       |   Combine query + retrieved context
|  Build prompt     |   into a single prompt
+-------------------+
    |
    v
+-------------------+
|  4. GENERATE      |   LLM generates answer
|  LLM response     |   grounded in context
+-------------------+
    |
    v
Answer to User
```

**Offline (indexing) phase:**
1. Split documents into chunks
2. Embed each chunk
3. Store embeddings in a vector database

**Online (query) phase:**
1. Embed the user query
2. Find the most similar chunks (nearest neighbor search)
3. Construct a prompt: "Given this context: {retrieved chunks}. Answer: {query}"
4. Feed to LLM and return the response

---
## 3. Component 1: Document Store and Chunking

### Why Chunk?

Documents can be very long (papers, books, manuals). We chunk them because:

- LLMs have a **context window limit** (e.g., 4K, 8K, 128K tokens)
- Smaller chunks = more **precise retrieval** (a paragraph about topic X vs an entire chapter)
- Better embedding quality: embedding a focused paragraph is more meaningful than embedding a whole document

### Chunking Strategies

| Strategy | Description | Pros | Cons |
|:-|:-|:-|:-|
| **Fixed-size** | Split every N characters/tokens | Simple, predictable | May split mid-sentence |
| **Sentence-based** | Split at sentence boundaries | Preserves meaning | Varying chunk sizes |
| **Paragraph-based** | Split at paragraph breaks | Natural boundaries | Paragraphs vary wildly in size |
| **Overlapping** | Fixed-size with M token overlap | Context continuity | More chunks to store |
| **Semantic** | Split where topic changes | Best semantic coherence | Complex to implement |

In [ ]:
# Demonstrate chunking strategies

sample_document = (
    "The Transformer architecture was introduced in 2017 by Vaswani et al. "
    "in the paper 'Attention Is All You Need'. It replaced recurrent layers "
    "with self-attention mechanisms, enabling parallel processing of sequences. "
    "This was a breakthrough in natural language processing. "
    "\n\n"
    "BERT, introduced by Google in 2018, uses the Transformer encoder for "
    "bidirectional language understanding. It was pretrained on masked language "
    "modeling and next sentence prediction tasks. BERT achieved state-of-the-art "
    "results on many NLP benchmarks. "
    "\n\n"
    "GPT, developed by OpenAI, uses the Transformer decoder for autoregressive "
    "text generation. GPT-2 demonstrated that large language models can generate "
    "remarkably coherent text. GPT-3 scaled to 175 billion parameters and showed "
    "impressive few-shot learning capabilities."
)


def chunk_by_paragraph(text):
    """Split text into chunks at paragraph boundaries (double newline)."""
    chunks = [p.strip() for p in text.split('\n\n') if p.strip()]
    return chunks


def chunk_fixed_size(text, chunk_size=100, overlap=20):
    """Split text into fixed-size character chunks with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start = end - overlap  # overlap with previous chunk
    return [c for c in chunks if c]  # remove empty chunks


def chunk_by_sentence(text, sentences_per_chunk=2):
    """Split text into chunks of N sentences."""
    # Simple sentence splitting on period + space
    sentences = re.split(r'(?<=[.!?])\s+', text.replace('\n', ' '))
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = ' '.join(sentences[i:i + sentences_per_chunk])
        chunks.append(chunk)
    return chunks


# Demonstrate each strategy
print("=" * 60)
print("PARAGRAPH CHUNKING")
print("=" * 60)
for i, chunk in enumerate(chunk_by_paragraph(sample_document)):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(f"  {chunk[:80]}..." if len(chunk) > 80 else f"  {chunk}")

print()
print("=" * 60)
print("FIXED-SIZE CHUNKING (100 chars, 20 overlap)")
print("=" * 60)
for i, chunk in enumerate(chunk_fixed_size(sample_document, 100, 20)):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(f"  {chunk[:80]}..." if len(chunk) > 80 else f"  {chunk}")

print()
print("=" * 60)
print("SENTENCE CHUNKING (2 sentences per chunk)")
print("=" * 60)
for i, chunk in enumerate(chunk_by_sentence(sample_document, 2)):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(f"  {chunk[:80]}..." if len(chunk) > 80 else f"  {chunk}")

---
## 4. Component 2: Embeddings and Vector Store

Each chunk is converted to a **dense vector** (embedding) so we can measure semantic similarity.

For this notebook, we use a simple **bag-of-words + TF-IDF-like** embedding approach that requires no external models. In production, you would use sentence-transformers or similar.

The vector store holds all chunk embeddings for fast retrieval.

In [ ]:
class SimpleEmbedder:
    """
    Simple TF-IDF-like text embedder (no external dependencies).
    
    Each document is represented as a normalized term-frequency vector
    weighted by inverse document frequency.
    """
    def __init__(self):
        self.vocab = {}
        self.idf = None
        self.vocab_size = 0
    
    def _tokenize(self, text):
        """Simple lowercase tokenization."""
        return re.findall(r'[a-z0-9]+', text.lower())
    
    def fit(self, documents):
        """Build vocabulary and compute IDF from a corpus."""
        # Build vocabulary
        all_tokens = set()
        for doc in documents:
            all_tokens.update(self._tokenize(doc))
        self.vocab = {token: i for i, token in enumerate(sorted(all_tokens))}
        self.vocab_size = len(self.vocab)
        
        # Compute IDF: log(N / df) where df = number of docs containing the term
        N = len(documents)
        df = np.zeros(self.vocab_size)
        for doc in documents:
            unique_tokens = set(self._tokenize(doc))
            for token in unique_tokens:
                if token in self.vocab:
                    df[self.vocab[token]] += 1
        
        self.idf = np.log((N + 1) / (df + 1)) + 1  # smoothed IDF
        print(f"Vocabulary size: {self.vocab_size}")
    
    def embed(self, text):
        """
        Embed a single text as a TF-IDF vector.
        
        Returns:
            Normalized embedding vector (vocab_size,)
        """
        tokens = self._tokenize(text)
        tf = np.zeros(self.vocab_size)
        for token in tokens:
            if token in self.vocab:
                tf[self.vocab[token]] += 1
        
        # Normalize TF
        if tf.sum() > 0:
            tf = tf / tf.sum()
        
        # TF-IDF
        tfidf = tf * self.idf
        
        # L2 normalize
        norm = np.linalg.norm(tfidf)
        if norm > 0:
            tfidf = tfidf / norm
        
        return tfidf
    
    def embed_batch(self, texts):
        """Embed multiple texts."""
        return np.array([self.embed(text) for text in texts])


print("SimpleEmbedder defined.")

---
## 5. Component 3: Retriever

The retriever takes a query embedding and finds the $k$ most similar document chunks using **cosine similarity** (or equivalently, dot product on normalized vectors).

In [ ]:
class SimpleRetriever:
    """
    Retrieves the most relevant document chunks for a query
    using cosine similarity on embeddings.
    """
    def __init__(self, embedder):
        self.embedder = embedder
        self.chunks = []
        self.chunk_embeddings = None
    
    def index(self, chunks):
        """Index a list of text chunks."""
        self.chunks = chunks
        self.chunk_embeddings = self.embedder.embed_batch(chunks)
        print(f"Indexed {len(chunks)} chunks (embedding shape: {self.chunk_embeddings.shape})")
    
    def retrieve(self, query, top_k=3):
        """
        Retrieve the top-k most relevant chunks for a query.
        
        Args:
            query: search query string
            top_k: number of chunks to retrieve
        Returns:
            List of (chunk_text, similarity_score) tuples
        """
        query_emb = self.embedder.embed(query)
        
        # Cosine similarity (vectors are already normalized)
        similarities = self.chunk_embeddings @ query_emb
        
        # Top-k indices
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        return [(self.chunks[i], similarities[i]) for i in top_indices]


print("SimpleRetriever defined.")

---
## 6. Component 4: Generator (Concept)

In a real RAG system, the generator is an LLM (GPT-4, LLaMA, etc.) that:

1. Receives an **augmented prompt** containing the query and retrieved context
2. Generates an answer **grounded in the context**

**Augmented prompt template:**

```
You are a helpful assistant. Answer the question based only on the provided context.
If the context does not contain the answer, say "I don't know."

Context:
---
{retrieved_chunk_1}
---
{retrieved_chunk_2}
---

Question: {user_query}

Answer:
```

Since we do not have an LLM in this notebook, we will **simulate** the generation step by constructing the prompt and showing what would be sent to the model.

In [ ]:
def build_rag_prompt(query, retrieved_chunks, max_context_chars=2000):
    """
    Build an augmented prompt for RAG.
    
    Args:
        query: user's question
        retrieved_chunks: list of (chunk_text, score) tuples
        max_context_chars: maximum total characters for context
    Returns:
        Formatted prompt string
    """
    # Build context from retrieved chunks
    context_parts = []
    total_chars = 0
    for chunk_text, score in retrieved_chunks:
        if total_chars + len(chunk_text) > max_context_chars:
            break
        context_parts.append(f"[Relevance: {score:.3f}]\n{chunk_text}")
        total_chars += len(chunk_text)
    
    context = "\n---\n".join(context_parts)
    
    prompt = (
        "You are a helpful assistant. Answer the question based only on the "
        "provided context. If the context does not contain the answer, say "
        "\"I don't know.\"\n\n"
        f"Context:\n---\n{context}\n---\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )
    return prompt


print("build_rag_prompt defined.")

---
## 7. Code: RAG Simulation

Let us put all the components together into a complete RAG pipeline simulation.

In [ ]:
# Knowledge base: a collection of documents about AI/ML topics
knowledge_base = [
    # Transformers
    "The Transformer architecture was introduced in 2017 by Vaswani et al. in the paper "
    "'Attention Is All You Need'. It uses self-attention mechanisms instead of recurrence, "
    "enabling parallel processing of entire sequences.",
    
    "Self-attention computes a weighted sum of all positions in a sequence, where weights "
    "are determined by the compatibility (dot product) between query and key vectors. "
    "This allows each token to directly attend to any other token.",
    
    "Multi-head attention runs multiple attention operations in parallel, each with "
    "different learned projections. This allows the model to attend to information "
    "from different representation subspaces at different positions.",
    
    # BERT
    "BERT (Bidirectional Encoder Representations from Transformers) was introduced by "
    "Google in 2018. It uses the Transformer encoder and is pretrained with masked "
    "language modeling (predicting randomly masked tokens) and next sentence prediction.",
    
    "BERT processes text bidirectionally, meaning it considers both left and right "
    "context simultaneously. This is in contrast to GPT, which only looks at left context. "
    "BERT is primarily used for understanding tasks like classification and QA.",
    
    # GPT
    "GPT (Generative Pre-trained Transformer) is a decoder-only Transformer developed "
    "by OpenAI. It is trained autoregressively to predict the next token. GPT-3 has "
    "175 billion parameters and demonstrated impressive few-shot learning.",
    
    "GPT models use causal (left-to-right) attention masking, meaning each token can "
    "only attend to previous tokens. This makes them natural for text generation tasks "
    "like writing, code generation, and dialogue.",
    
    # Training
    "Transfer learning in NLP involves pretraining a model on a large corpus (like "
    "Wikipedia) and then fine-tuning it on a specific downstream task with a smaller "
    "labeled dataset. This approach drastically reduces the data needed for good performance.",
    
    "The learning rate for fine-tuning pretrained Transformers is typically much smaller "
    "than for training from scratch (2e-5 to 5e-5 vs 1e-3). A warmup schedule is commonly "
    "used to prevent destroying the pretrained weights in early training steps.",
    
    # Embeddings
    "Word embeddings like Word2Vec and GloVe map words to dense vectors where semantic "
    "similarity corresponds to vector proximity. However, they produce a single static "
    "embedding per word regardless of context.",
    
    "Contextual embeddings from models like BERT produce different vectors for the same "
    "word depending on its context. For example, 'bank' in 'river bank' gets a different "
    "embedding than 'bank' in 'bank account'.",
    
    # RAG itself
    "Retrieval-Augmented Generation (RAG) combines retrieval systems with language models. "
    "Instead of relying solely on the model's parametric knowledge, RAG retrieves relevant "
    "documents and includes them in the prompt for more accurate, grounded responses.",
]

print(f"Knowledge base: {len(knowledge_base)} chunks")
for i, chunk in enumerate(knowledge_base):
    print(f"  Chunk {i:2d}: {chunk[:70]}...")

In [ ]:
# Build the RAG pipeline
set_global_seed(42)

# Step 1: Create embedder and fit on knowledge base
embedder = SimpleEmbedder()
embedder.fit(knowledge_base)

# Step 2: Create retriever and index chunks
retriever = SimpleRetriever(embedder)
retriever.index(knowledge_base)

print("\nRAG pipeline ready!")

In [ ]:
# Step 3: Query the RAG system

queries = [
    "What is self-attention?",
    "How is BERT different from GPT?",
    "What learning rate should I use for fine-tuning?",
    "What is RAG and why is it useful?",
    "How do contextual embeddings work?",
]

for query in queries:
    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)
    
    # Retrieve relevant chunks
    results = retriever.retrieve(query, top_k=3)
    
    print("\nRetrieved chunks:")
    for rank, (chunk, score) in enumerate(results, 1):
        print(f"  {rank}. [sim={score:.4f}] {chunk[:80]}...")
    
    # Build augmented prompt
    prompt = build_rag_prompt(query, results)
    print(f"\nAugmented prompt ({len(prompt)} chars):")
    print("-" * 40)
    # Show truncated prompt
    lines = prompt.split('\n')
    for line in lines[:5]:
        print(f"  {line}")
    print("  ...")
    for line in lines[-4:]:
        print(f"  {line}")
    print()
    print("(In a real RAG system, this prompt would be sent to an LLM for generation.)")
    print()

In [ ]:
# Visualize retrieval relevance
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, query in enumerate(queries[:6]):
    if idx >= len(queries):
        break
    row, col = idx // 3, idx % 3
    ax = axes[row, col]
    
    # Get similarities for all chunks
    query_emb = embedder.embed(query)
    similarities = retriever.chunk_embeddings @ query_emb
    
    # Sort for display
    sorted_indices = np.argsort(similarities)[::-1]
    sorted_sims = similarities[sorted_indices]
    
    colors = ['tomato' if i < 3 else 'steelblue' for i in range(len(sorted_sims))]
    ax.barh(range(len(sorted_sims)), sorted_sims, color=colors, alpha=0.7)
    ax.set_yticks(range(len(sorted_sims)))
    ax.set_yticklabels([f"Chunk {i}" for i in sorted_indices], fontsize=8)
    ax.set_xlabel('Cosine Similarity')
    ax.set_title(f"'{query}'", fontsize=10)
    ax.invert_yaxis()

# Remove unused subplot if odd number of queries
if len(queries) < 6:
    for idx in range(len(queries), 6):
        row, col = idx // 3, idx % 3
        axes[row, col].set_visible(False)

plt.suptitle('Retrieval Relevance Scores\n(Red = top-3 retrieved chunks)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Vector Databases (Concept)

For production RAG systems, you need a **vector database** -- a specialized database optimized for storing and querying high-dimensional vectors.

### Why Not Just Use NumPy?

Our simple approach computes cosine similarity with **all** stored vectors (brute force). This is:
- $O(N \cdot D)$ per query, where $N$ = number of chunks, $D$ = embedding dimension
- Fine for hundreds or even thousands of chunks
- **Too slow** for millions or billions of chunks

### How Vector Databases Solve This

Vector databases use **approximate nearest neighbor (ANN)** algorithms:

| Algorithm | Description |
|:-|:-|
| **HNSW** (Hierarchical Navigable Small World) | Graph-based; builds a navigable graph of vectors |
| **IVF** (Inverted File Index) | Clusters vectors; only searches relevant clusters |
| **PQ** (Product Quantization) | Compresses vectors for memory efficiency |

### Popular Vector Databases

| Database | Type | Notes |
|:-|:-|:-|
| **FAISS** (Facebook) | Library | Fast, in-memory, good for experiments |
| **Pinecone** | Managed service | Cloud-hosted, easy to use |
| **Weaviate** | Self-hosted/cloud | Supports hybrid search |
| **Chroma** | Library | Lightweight, Python-native |
| **Milvus** | Self-hosted | Scalable, open-source |
| **pgvector** | PostgreSQL extension | Adds vector search to PostgreSQL |

### Trade-offs: Accuracy vs Speed

ANN algorithms trade off **recall** (finding the true nearest neighbors) for **speed**:

- More accurate = slower (closer to brute force)
- Faster = may miss some relevant results
- Most production systems achieve 95-99% recall with 10-100x speedup

In [ ]:
# Illustrate the scaling problem
import time

set_global_seed(42)

dims = 384  # typical sentence-transformer dimension
query = np.random.randn(dims)
query = query / np.linalg.norm(query)

print(f"{'N chunks':>12} | {'Brute-force time (ms)':>22} | {'Note':>30}")
print("-" * 72)

for n in [100, 1_000, 10_000, 100_000]:
    # Create random embeddings
    embeddings = np.random.randn(n, dims)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / norms
    
    # Time brute-force search
    start = time.time()
    for _ in range(10):  # average over 10 runs
        sims = embeddings @ query
        top_k = np.argsort(sims)[-5:][::-1]
    elapsed = (time.time() - start) / 10 * 1000  # ms
    
    note = ""
    if n <= 1000:
        note = "Fine with brute force"
    elif n <= 10_000:
        note = "Still OK, but consider ANN"
    else:
        note = "Use a vector database"
    
    print(f"{n:>12,} | {elapsed:>22.2f} | {note:>30}")

print("\nFor millions of chunks, brute force takes seconds per query.")
print("ANN indexes (HNSW, IVF) bring this down to milliseconds.")

---
## 9. Exercise: Design a RAG Pipeline on Paper

**Task:** Design a RAG system for a **customer support chatbot** for a software product. Answer the following questions:

1. **Document sources:** What documents would you include in the knowledge base?
   - (e.g., user manual, FAQ, release notes, API docs, support tickets)

2. **Chunking strategy:** How would you chunk each document type?
   - What chunk size?
   - Overlap or no overlap?
   - Different strategies for different document types?

3. **Embedding model:** What embedding model would you use?
   - Consider: quality, speed, dimension, cost

4. **Retrieval:** How many chunks would you retrieve? How would you handle:
   - Ambiguous queries?
   - No relevant chunks found?
   - Chunks from multiple document types?

5. **Prompt template:** Write a prompt template for the customer support use case.

6. **Evaluation:** How would you measure the quality of your RAG system?

```
Write your design in this cell or on paper:

1. Document sources:
   - ...

2. Chunking strategy:
   - ...

3. Embedding model:
   - ...

4. Retrieval:
   - ...

5. Prompt template:
   - ...

6. Evaluation:
   - ...
```

In [ ]:
# Try the exercise yourself before looking at the solution!





### Solution

**1. Document sources:**
- User manual / product documentation (most important)
- FAQ pages (common issues, already in Q&A format)
- Release notes (for version-specific questions)
- API documentation (for developer-facing questions)
- Resolved support tickets (real-world problems and solutions)
- Knowledge base articles written by support staff

**2. Chunking strategy:**
- User manual: paragraph-based chunking (~200-300 tokens per chunk), with 50-token overlap to preserve context across chunk boundaries
- FAQ: each Q&A pair is one chunk (natural boundary)
- API docs: one chunk per endpoint/function
- Support tickets: one chunk per ticket (question + resolution)
- Different document types may benefit from different strategies

**3. Embedding model:**
- `all-MiniLM-L6-v2` from sentence-transformers (384 dimensions, fast, good quality)
- For higher quality: `all-mpnet-base-v2` (768 dimensions, slower but more accurate)
- Consider: OpenAI `text-embedding-3-small` if budget allows and API latency is acceptable

**4. Retrieval:**
- Retrieve top-5 chunks by default
- Ambiguous queries: retrieve more chunks (top-10) and let the LLM decide relevance
- No relevant chunks: set a minimum similarity threshold (e.g., 0.3); if all below, respond with "I don't have information on that, please contact support"
- Multiple types: include metadata (source type) in the retrieved context so the LLM can attribute its answer

**5. Prompt template:**
```
You are a helpful customer support agent for [Product Name].
Answer the customer's question using ONLY the provided documentation.
If the answer is not in the documentation, say "I'll escalate this to our support team."
Be concise and friendly.

Documentation:
---
{retrieved_chunks}
---

Customer question: {query}

Response:
```

**6. Evaluation:**
- **Retrieval quality:** Measure recall@k (are the correct chunks retrieved?)
- **Answer quality:** Human evaluation of correctness, helpfulness, and faithfulness to source
- **Hallucination rate:** % of answers that contain information not in the retrieved context
- **User satisfaction:** Thumbs up/down from users, follow-up question rate
- **Automated metrics:** BERTScore, ROUGE between generated and gold answers (if available)

---
## 10. Common Mistakes & Debugging Tips

**1. Chunks too large or too small**
- Too large: retrieval is imprecise (retrieved context includes irrelevant information)
- Too small: chunks lack context (a sentence in isolation may not be meaningful)
- Sweet spot: 100-300 tokens with some overlap (experiment for your use case)

**2. Ignoring chunk boundaries**
- Fixed-size chunking can split mid-sentence or mid-paragraph
- Use overlap or sentence-aware splitting to preserve context

**3. Embedding model mismatch**
- Query and document embeddings **must** use the same model
- Embeddings from different models live in different vector spaces

**4. Not including metadata**
- Store metadata with chunks (source document, page number, timestamp)
- This helps with attribution, filtering, and debugging

**5. Prompt too long (exceeds context window)**
- If you retrieve too many chunks, the augmented prompt may exceed the LLM's context window
- Solution: limit the number of chunks, summarize long chunks, or use a model with a larger context window

**6. Retrieval returns irrelevant chunks**
- Set a minimum similarity threshold to filter low-quality matches
- Consider hybrid search: combine semantic (embedding) search with keyword (BM25) search

**7. The LLM ignores the context**
- Prompt engineering matters: explicitly instruct the model to use only the provided context
- Some models are better at following instructions than others
- Consider using the context as the system prompt rather than user message

**8. Not updating the knowledge base**
- RAG is only as good as its knowledge base
- Schedule regular re-indexing when documents change
- Use a pipeline that detects and re-embeds updated documents

---

**Congratulations!** You have completed the Transformers and Modern NLP notebook series.

**Summary of the series:**
1. Self-Attention: Intuition and Math
2. Transformer Architecture: Encoder-Decoder
3. Training a Small Transformer for Text Classification
4. Fine-Tuning with HuggingFace Transformers (Optional)
5. Embeddings and Semantic Search (Optional)
6. Prompting & Inference: Temperature, Top-K, Top-P (Optional)
7. RAG: Concept Only (Optional) -- **this notebook**